In [ ]:
# ==========================================
# NLGCL Training on Kaggle (All-in-One)
# ==========================================

import os
import shutil
import pandas as pd
import numpy as np

# 1. Install Dependencies
print(">>> Installing Dependencies...")
# Pin numpy < 2.0 because RecBole/PyTorch legacy often breaks with NumPy 2.0
# Pin colorama >= 0.4.6 to satisfy bayesian-optimization
!pip install -q recbole colorlog tensorboard "colorama>=0.4.6" pyyaml pandas hyperopt==0.2.5 gdown "numpy<2.0"

# 2. Install PyTorch Geometric and related libraries
# Note: Kaggle usually runs on Linux with CUDA.
import torch

def format_pytorch_version(version):
    return version.split('+')[0]

TORCH_version = torch.__version__
CUDA_version = torch.version.cuda

print(f"Current Torch version: {TORCH_version}")
print(f"Current CUDA version: {CUDA_version}")

# If the Torch version is too new (e.g. 2.5+ or nightly) or unsupported by PyG wheels, 
# we might need to downgrade or compile from source. 
# However, usually just trying to install compatible scattered/sparse works if we don't force binary wheels for the wrong version.
# BUT, to be safe and fast, let's install a known stable configuration if wheels fail.
# Here we try strict install of wheels for current torch. If it fails (like for 2.9.0?), we fallback or user might need to downgrade.
# For now, let's REMOVE pyg_lib (optional and problematic) and use the flexible install.

TORCH = format_pytorch_version(TORCH_version)
CUDA = "cu" + CUDA_version.replace(".", "")

print(f"Installing for Torch {TORCH} and CUDA {CUDA}")

# Attempt to install PyG dependencies. 
# We remove pyg_lib as it often causes "No matching distribution" errors on newer setups.
# We also allow pip to find compatible versions if the direct link fails.
try:
    !pip install -q torch-geometric
    !pip install -q torch_scatter torch_sparse torch_cluster torch_spline_conv -f https://data.pyg.org/whl/torch-{TORCH}+{CUDA}.html
except:
    print("Standard wheel install failed. Trying without -f link (building from source or finding other wheels)...")
    !pip install -q torch-geometric
    !pip install -q torch_scatter torch_sparse torch_cluster torch_spline_conv

# 3. Clone Repository
if not os.path.exists("NLGCL"):
    print(">>> Cloning Repository...")
    !git clone https://github.com/yangzeha/NLGCL.git
    %cd NLGCL
else:
    print(">>> Repository already exists.")
    if os.getcwd().split(os.sep)[-1] != "NLGCL":
        %cd NLGCL

# 4. Patch Missing Files (Important step to fix the ImportError)
print(">>> Patching Missing Modules...")
# List of files that were missing in the local environment and caused errors
# We create dummy classes for them so the __init__.py imports don't fail.
missing_modules = [
    ("lightgcn", "LightGCN"),
    ("hmlet", "HMLET"),
    ("ncl", "NCL"),
    ("ngcf", "NGCF"),
    ("sgl", "SGL"),
    ("lightgcl", "LightGCL"),
    ("simgcl", "SimGCL"),
    ("xsimgcl", "XSimGCL"),
    ("directau", "DirectAU"),
    ("ssl4rec", "SSL4REC"),
    ("dccf", "DCCF"),
    ("l2cl", "L2CL")
]

base_path = "recbole_gnn/model/general_recommender"
if not os.path.exists(base_path):
    # Ensure path exists (in case we are not in the right dir, but we did cd)
    pass

for module_name, class_name in missing_modules:
    file_path = os.path.join(base_path, f"{module_name}.py")
    # Only create if it doesn't exist (it likely won't)
    if not os.path.exists(file_path):
        # print(f"Creating dummy file for: {file_path}")
        with open(file_path, 'w') as f:
            f.write(f"class {class_name}:\n    pass\n")

# 5. Dataset Preparation (Tenrec QB-Video)
print(">>> Preparing QB-Video Dataset...")
dataset_name = "QB-video"
data_dir = f"dataset/{dataset_name}"
if not os.path.exists(data_dir):
    os.makedirs(data_dir)

# Download QB-Video from Tenrec (Google Drive)
# File ID found from Tenrec GitHub/Papers: 1R1JhdT9CHzT3qBJODz09pVpHMzShcQ7a
file_id = '1R1JhdT9CHzT3qBJODz09pVpHMzShcQ7a'
output_file = os.path.join(data_dir, 'QB-video.csv')
inter_file = os.path.join(data_dir, f"{dataset_name}.inter")

if not os.path.exists(output_file) and not os.path.exists(inter_file):
    print("Downloading QB-Video dataset...")
    import gdown
    url = f'https://drive.google.com/uc?id={file_id}'
    gdown.download(url, output_file, quiet=False)

# Convert to RecBole Format (.inter)
if os.path.exists(output_file) and not os.path.exists(inter_file):
    print("Converting to RecBole .inter format...")
    try:
        # Read CSV (Tenrec format usually: user_id, item_id, click, like, follow, ...)
        df = pd.read_csv(output_file)
        
        # Identify user/item columns typically used in Tenrec
        # User ID often 'user_id', Item ID often 'video_id' or 'item_id'
        user_col = next((c for c in df.columns if 'user' in c.lower()), None)
        item_col = next((c for c in df.columns if 'item' in c.lower() or 'video' in c.lower()), None)
        
        if user_col and item_col:
            print(f"Mapping {user_col} -> user_id:token, {item_col} -> item_id:token")
            # Create new dataframe with RecBole column names
            new_df = pd.DataFrame()
            new_df['user_id:token'] = df[user_col]
            new_df['item_id:token'] = df[item_col]
            
            # RecBole needs string/token IDs usually, or we can use int if token is removed, 
            # but user_id:token is standard for IDs.
            
            # Drop duplicates
            new_df = new_df.drop_duplicates()
            
            # Save as .inter
            new_df.to_csv(inter_file, index=False, sep='\t')
            print(f"Saved {inter_file} with {len(new_df)} interactions.")
        else:
            print("Error: Could not identify user_id/item_id columns.")
            print(f"Columns found: {df.columns.tolist()}")

    except Exception as e:
        print(f"Error converting dataset: {e}")

# 6. Run Training
print(">>> Starting Training...")
# QB-Video settings based on author's recommendation:
# n_layers = 4, reg_weight = 0.0001, cl_temp = 0.2, alpha = 0.6, cl_reg = 5e-5
!python main.py --dataset QB-video --model NLGCL --n_layers=4 --reg_weight=1e-4 --cl_temp=0.2 --alpha=0.6 --cl_reg=5e-5